# Subreddit metadata from PRAW (v4 sample, pre-accusation subreddits)

For classifying subreddits by what they say they discuss rather than by name. Looks up every subreddit the v4 sample used
before first accusation (from `sample_1000_v4_output/`) with `reddit.info(fullnames=...)`, 100 per request, using the
`subreddit_id` stored on the archived posts/comments.

Kept per subreddit: Reddit's NSFW label (`over18`), short bio (`public_description`), sidebar (`description`), `title`,
Reddit's `advertiser_category`, type, subscribers, quarantine, creation time. Banned subreddits come back with almost no
fields; `status` records what came back. Output: `subreddit_review_output/subreddit_praw_meta_v4.parquet` (never overwritten).

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import time
import praw
import prawcore
from dotenv import load_dotenv
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Folders, the .env file holding the PRAW credentials (never written here), and the subreddit fields to keep.
SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
ENV_PATH = '/home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env'
KEEP = ['display_name', 'name', 'title', 'public_description', 'description', 'over18', 'subreddit_type', 'subscribers',
        'quarantine', 'advertiser_category', 'content_category', 'lang', 'submission_type', 'allow_images', 'allow_videos',
        'created_utc']
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Every subreddit (name and fullname id) the v4 sample used before its first accusation.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
parts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit', 'subreddit_id'])
    parts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])][['subreddit', 'subreddit_id']])
subs = pd.concat(parts).drop_duplicates().reset_index(drop=True)
assert subs['subreddit'].is_unique and subs['subreddit_id'].notna().all()
print(f'{len(subs):,} subreddits to look up')

In [ ]:
# Log in to the Reddit API with credentials from the .env file.
load_dotenv(ENV_PATH)
reddit = praw.Reddit(client_id=os.getenv('praw_script'), client_secret=os.getenv('praw_key'),
                     username=os.getenv('login_name'), password=os.getenv('password'),
                     user_agent=os.getenv('user_agent'), ratelimit_seconds=300)

# Look up subreddits 100 at a time (reddit.info takes up to 100 fullnames per call).
rows = []
fullnames = subs['subreddit_id'].tolist()
for start in range(0, len(fullnames), 100):
    batch = fullnames[start:start + 100]
    retry_count = 0
    # Retry a failed batch up to 10 times, waiting longer each time.
    while True:
        try:
            got = list(reddit.info(fullnames=batch))
            break
        except (prawcore.exceptions.PrawcoreException, praw.exceptions.PRAWException) as e:
            retry_count += 1
            if retry_count >= 10:
                raise
            time.sleep(10 * retry_count)
    fetched = int(time.time())
    # Keep the chosen fields, how many fields came back, and when the batch was fetched.
    for sub in got:
        v = vars(sub)
        rows.append({**{k: v.get(k) for k in KEEP}, 'n_fields': sum(not k.startswith('_') for k in v), 'fetched_utc': fetched})
    if start % 2000 == 0:
        print(f'{start + len(batch):,} / {len(fullnames):,}')
meta = pd.DataFrame(rows)
print(f'returned {len(meta):,} of {len(subs):,}')

In [ ]:
# Join the results back onto the subreddit list and give each subreddit a status:
# not returned, banned/removed (no metadata), private, user profile (u_ prefix), or ok.
meta = subs.merge(meta.rename(columns={'name': 'subreddit_id'}), on='subreddit_id', how='left', validate='one_to_one')
meta['status'] = np.select(
    [meta['n_fields'].isna(), meta['subscribers'].isna() & meta['over18'].isna(), meta['subreddit_type'].eq('private'),
     meta['subreddit'].str.startswith('u_')],
    ['not returned', 'banned or removed (no metadata)', 'private', 'user profile'], default='ok')
for c in ['title', 'public_description', 'description', 'advertiser_category', 'content_category', 'lang', 'submission_type', 'subreddit_type']:
    meta[c] = meta[c].astype('string')
# Status counts, status by NSFW flag, and Reddit's advertiser category for live subreddits.
display(meta['status'].value_counts().to_frame())
display(pd.crosstab(meta['status'], meta['over18'].astype(str)))
meta.loc[meta['status'].eq('ok'), 'advertiser_category'].fillna('<none>').replace('', '<blank>').value_counts().head(25).to_frame()

In [ ]:
# Save the metadata table.
save_new(meta, 'subreddit_praw_meta_v4.parquet')